# The Carolina Way: UNC's Enduring Tournament Legacy


# Introduction
    The month of March is always exciting in sports. It's the start of March Madness! D1 basketball teams across the country compete on the national stage for a chance to win the coveted NCAA D1 Men's basketball tournament. What makes it exciting are the upsets and rivalries that makes the tournament so captivating. In college basketball, the basketball rivalry that always stands out is UNC vs Duke. Personally, I'm a UNC fan as my family had relatives and even my older sister attend UNC. I would say UNC has a very good track record when it comes to March Madness with it's numerous trips to the Round of 64 all the way to the Final Four and 6 Championship Titles held. The past few years with Hubert Davis as Head Coach, the team has had its rough patches so the question I want to explore is "Will UNC be the champions of the next NCAA March Madness Tournament?"

# The DataSet

    The data set I chose is a file found from Kaggle: https://www.kaggle.com/datasets/andrewsundberg/college-basketball-dataset?select=cbb25.csv. Although there are team-level statistics from the past 12 years for all NCAA D1 Men's Basketball Team I wanted to focus on the team-level statistics from the most recent season, 2024-2025. 
    
    From the dataset features provided below, I want to explore training a classification model to predict whether a team like UNC qualifies for the tournament and how far they might advance.

### DataSet Features

    Basic Performance Stats
        G --> games played
        W --> wins (raw total wins)

    Efficiency metrics 
        ADJOE --> adjusted offensive effciency (points per 100 possessions, offense)
        ADJDE --> adjusted defnsive efficiency (point allowed per 100 possessions, defense)

    Shooting & Scoring Efficiency
        2P_O --> 2-point shooting % for *team*
        3P_0 --> 3-point shooting % for a *team*

    Possession & Rebounding
        TOR --> Turnover rate (offense)
        ORB --> Offensive rebound rate
        DRB --> Defensive rebound rate

    Tournament Related
        SEED --> NCAA tournament seed (1-16 teams that qualified; null for teams that didn't)



In [65]:
import numpy as np
import pandas as pd
import seaborn as sb
import matplotlib as mb
import matplotlib.pyplot as plt



In [66]:
#Load the Dataset
unc_df = pd.read_csv("../datasets/cbb25.csv")

In [67]:
# Basic information about the dataset
print("Dataset Info: ")
unc_df.info()

Dataset Info: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 364 entries, 0 to 363
Data columns (total 25 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   RK       364 non-null    int64  
 1   Team     364 non-null    object 
 2   CONF     364 non-null    object 
 3   G        364 non-null    int64  
 4   W        364 non-null    int64  
 5   ADJOE    364 non-null    float64
 6   ADJDE    364 non-null    float64
 7   BARTHAG  364 non-null    float64
 8   EFG_O    364 non-null    float64
 9   EFG_D    364 non-null    float64
 10  TOR      364 non-null    float64
 11  TORD     364 non-null    float64
 12  ORB      364 non-null    float64
 13  DRB      364 non-null    float64
 14  FTR      364 non-null    float64
 15  FTRD     364 non-null    float64
 16  2P_O     364 non-null    float64
 17  2P_D     364 non-null    float64
 18  3P_O     364 non-null    float64
 19  3P_D     364 non-null    float64
 20  3PR      364 non-null    float64
 21  3

# Pre Processing

#### For Step 1, I imported the libraries and Python tools I'll need 
#### For Step 2, I loaded the dataset and reads the CSV into memory as a table "unc_df"
#### For Step 3, I created a target value that makes a binary column: 1 if team has a SEED (made the tournament), 0 otherwise
#### For Step 4, I identified categorical vs numeric features to tell the pipeline which columns are categories (what conference the team is in) and which are numerical stats (ADJOE & ADJDE)
#### For Step 5, I defined my transformations: standardized numeric columns and one-hot encode conferences into dummy variables. One-hot encoding turns each conference into its own column of 0s and 1s. 
#### For Step 6, I split the data so I can train on 80% of the teams and evaluate on 20% the model hasn't seen.
#### For Step 7, I created a pipeline consisting of only the proprocessor. 


In [68]:
# ======================
# Step 1: Import Libraries
# ======================
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


In [69]:
# ======================
# Step 2: Load the dataset
# ======================
unc_df = pd.read_csv("../datasets/cbb25.csv")


In [70]:
# ======================
# Step 3: Define target variable
# ======================
# Binary target: 1 = tournanent, 0 = non-tournament
unc_df["Tournment"] = unc_df["SEED"].notnull().astype(int)



In [71]:
# ======================
# Step 4: Identify Feature Types
# ======================
# Categorical
cat_features = ["CONF"]

# Numerical (all the other columns)
num_features = unc_df.drop(columns=["Tournment", "CONF"]).columns.tolist()

In [72]:
# ======================
# Step 5: Pre processing
# ======================
#One-hot encode categorical and numerics
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(drop = "first"), cat_features)
    ]
)

In [73]:
# ======================
# Step 6: Train-test split
# ======================
X = unc_df.drop(columns = "Tournment", axis = 1)
y = unc_df["Tournment"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

In [74]:
# ======================
# Step 7: Final preprocessed pipeline
# ======================
pipeline = Pipeline(steps = [
    ("preprocessor", preprocessor)    
])